<a href="https://colab.research.google.com/github/2403a54116-blip/sreenidhi-NLP/blob/main/Lab_Assignment_14_4_2403a54116.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

In [37]:
import pandas as pd

# Read the CSV file, assuming no header, and the first column is the label.
df = pd.read_csv('/content/sample_data/mnist_train_small.csv', header=None)

# The first column is the label (digit), and the rest are features (pixel values).
labels_data = df.iloc[:, 0]
features_data = df.iloc[:, 1:]

# For MNIST, these are numerical features, not text. CountVectorizer is not appropriate.
# Assign X to features_data and y to labels_data for now.
X = features_data.values # Convert DataFrame to NumPy array
y = labels_data.values   # Convert Series to NumPy array

In [38]:
print(X)
print(y)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
[6 5 7 ... 2 9 5]


In [39]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


In [40]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [41]:
train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=2, shuffle=True)
test_loader = DataLoader(TextDataset(X_test, y_test), batch_size=2)

In [42]:
class CNNModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        kernels=4
        kernel_size=3
        self.conv = nn.Conv1d(1, kernels, kernel_size)

        pool_size=2
        self.pool = nn.MaxPool1d(pool_size)

        # Calculate flattened size
        flattened_size = kernels * ((input_size - pool_size)//2)

        # Hidden layer
        hidden_neurons=8
        self.fc1 = nn.Linear(flattened_size, hidden_neurons)

        # Output layer
        output_neurons=10
        self.fc2 = nn.Linear(hidden_neurons, output_neurons)

    def forward(self, x):
        x = x.unsqueeze(1)              # (batch, 1, features)
        x = torch.relu(self.conv(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)       # Flatten

        x = torch.relu(self.fc1(x))     # Hidden layer + activation
        x = self.fc2(x)                 # Output layer

        return x

In [43]:
model = CNNModel(input_size=X.shape[1])

In [44]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [45]:
for epoch in range(10):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f})")

Epoch 1, Loss: 2.5175)
Epoch 2, Loss: 2.2004)
Epoch 3, Loss: 2.3075)
Epoch 4, Loss: 2.2870)
Epoch 5, Loss: 2.3635)
Epoch 6, Loss: 2.2552)
Epoch 7, Loss: 2.3249)
Epoch 8, Loss: 2.2353)
Epoch 9, Loss: 2.3041)
Epoch 10, Loss: 2.3493)


In [46]:
for epoch in range(10):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch) # No squeeze needed, CrossEntropyLoss expects logits
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f})")

Epoch 1, Loss: 2.2442)
Epoch 2, Loss: 2.2403)
Epoch 3, Loss: 2.3091)
Epoch 4, Loss: 2.3430)
Epoch 5, Loss: 2.3528)
Epoch 6, Loss: 2.3674)
Epoch 7, Loss: 2.3449)
Epoch 8, Loss: 2.3473)
Epoch 9, Loss: 2.2815)
Epoch 10, Loss: 2.3423)


In [47]:
y_pred = []
y_actual = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        # For multi-class classification, get the index of the max logit
        preds = torch.argmax(output, dim=1)
        y_pred.extend(preds.tolist())
        y_actual.extend(y_batch.tolist())


# Convert actual_labels from float to int if needed by sklearn metrics
y_actual = [int(y) for y in y_actual]
print(y_pred)

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [48]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("\nConfusion Matrix:")
print(confusion_matrix(y_actual, y_pred))

print("\nClassification Report:")
print(classification_report(y_actual, y_pred))

print("\nAccuracy Score:", accuracy_score(y_actual, y_pred))


Confusion Matrix:
[[  0 377   0   0   0   0   0   0   0   0]
 [  0 457   0   0   0   0   0   0   0   0]
 [  0 383   0   0   0   0   0   0   0   0]
 [  0 428   0   0   0   0   0   0   0   0]
 [  0 390   0   0   0   0   0   0   0   0]
 [  0 342   0   0   0   0   0   0   0   0]
 [  0 422   0   0   0   0   0   0   0   0]
 [  0 423   0   0   0   0   0   0   0   0]
 [  0 367   0   0   0   0   0   0   0   0]
 [  0 411   0   0   0   0   0   0   0   0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       377
           1       0.11      1.00      0.21       457
           2       0.00      0.00      0.00       383
           3       0.00      0.00      0.00       428
           4       0.00      0.00      0.00       390
           5       0.00      0.00      0.00       342
           6       0.00      0.00      0.00       422
           7       0.00      0.00      0.00       423
           8       0.00      0.00      

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
